# A Logic Puzzle

The following exercise is taken from the book 
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by 
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full fullnames of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}x\texttt{,}z\texttt{>}$ for any female first name $x$ and any last name $z$ expresses
  that the lady with first name $x$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

In [1]:
import { RecursiveSet as Set, Value, Tuple, flatMap } from 'recursive-set';
import { solve } from './06-Davis-Putnam';

In [2]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [3]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

In [4]:
const FirstMale   = set("Erich",  "Fritz", "Gustav", "Heiner");
const FirstFemale = set("Inge",  "Josefa", "Klara",  "Luise");
const SurNames    = set("Amann", "Bemann", "Cemann", "Demann");

Instead of loading an external string parser, we will define our Propositional Logic types and a few robust helper functions directly in TypeScript. This provides better type-safety and leverages `recursive-set`'s value semantics for Conjunctive Normal Form (CNF) clauses.

In [5]:
type Variable = string;
type Literal  = Variable | Tuple<['¬', Variable]>;
type Clause   = Set<Literal>;
type Clauses  = Set<Clause>;

function complement(l: Literal): Literal {
    if (typeof l == 'string') { return new Tuple('¬', l); }
    return l.get(1);
}

function not(p: Variable): Literal {
    return tpl('¬', p);
}

The function `makeVar(f, x, y)` creates a propositional variable of the form `f<x,y>`.

In [6]:
function makeVar(f: string, x: string, y: string): Variable {
    return `${f}<${x},${y}>`;
}

In [7]:
makeVar('Married', 'Heiner', 'Klara')

Married<Heiner,Klara>


Given a set of propositional variables $S$, the function `atMostOne(S)` computes a set of clauses expressing the fact that at most one of the variables of $S$ is `True`.

In [8]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    return S.cartesianProduct(S)
            .filterMap(([p, q]) => p != q,
                       ([p, q]) => set(not(p), not(q)));
}

Given a set of propositional variables $S$, the function `atLeastOne(S)` computes a set of clauses expressing the fact that at least one of the variables of $S$ is `True`.

In [9]:
function atLeastOne(S: Set<Variable>): Clauses {
    return set(S);
}

$S$ is a set of propositional variables. The expression `exactlyOne(S)` creates a set of clauses.  This set expresses the fact that exactly one of the variables in the set $S$ is true.

In [10]:
function exactlyOne(S: Set<Variable>): Clauses {
    return atMostOne(S).union(atLeastOne(S));
}

In [11]:
exactlyOne(set('a', 'b', 'c'))

{{(¬, a), (¬, b)}, {(¬, a), (¬, c)}, {(¬, b), (¬, c)}, {a, b, c}}


For two sets $A$ and $B$ that have the same number of elements and a function symbol $f$, the procedure `bijective(A, B)` computes a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

In [32]:
function bijective(A: Set<string>, B: Set<string>, f: string): Clauses {
    let clauses = flatMap(A, a => exactlyOne(B.map(b => makeVar(f, a, b))));
    return clauses.union(flatMap(B, b => exactlyOne(A.map(a => makeVar(f, a, b)))));
}

In [33]:
bijective(set('a', 'b'), set('x', 'y'), 'f')

{{(¬, f<a,x>), (¬, f<a,y>)}, {(¬, f<a,x>), (¬, f<b,x>)}, {(¬, f<a,y>), (¬, f<b,y>)}, {(¬, f<b,x>), (¬, f<b,y>)}, {f<a,x>, f<a,y>}, {f<a,x>, f<b,x>}, {f<a,y>, f<b,y>}, {f<b,x>, f<b,y>}}


We introduce standard logical connectives (`implies`, `exclusiveOr`, `equivalence`, `impliesAnd`) as helper functions that directly yield sets of CNF clauses, bypassing the need for a complex string parser.

In [34]:
function implies(a: Literal, b: Literal): Clauses {
    // a → b ≡ ¬a ∨ b
    return set(set(complement(a), b));
}

function exclusiveOr(a: Literal, b: Literal): Clauses {
    // a ↔ ¬b ≡ (a ∨ b) ∧ (¬a ∨ ¬b)
    return set(set(a, b), set(complement(a), complement(b)));
}

function equivalence(a: Literal, b: Literal): Clauses {
    // a ↔ b ≡ (¬a ∨ b) ∧ (a ∨ ¬b)
    return set(set(complement(a), b),set(a, complement(b)));
}

function impliesAnd(a: Literal, b: Literal, c: Literal): Clauses {
    // a ∧ b → c ≡ ¬a ∨ ¬b ∨ c
    return set(set(complement(a), complement(b), c));
}

In [35]:
exclusiveOr('p', 'q')

{{(¬, p), (¬, q)}, {p, q}}


Given a set of male first names `FirstMale`, a set of female first names `FirstFemale`, and a set of surnames `Surnames`,
the function `consistentNames(FirstMale, FirstFemale, Surnames)` returns a set of clauses that ensures that if the man
`x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [36]:
function consistentNames(FirstMale:   Set<string>, 
                         FirstFemale: Set<string>, 
                         SurNames:    Set<string>): Clauses 
{
    return flatMap(FirstMale, x =>
        flatMap(FirstFemale, y =>
            flatMap(SurNames, z => {
                const a = makeVar('MaleName',   x, z);
                const b = makeVar('FemaleName', y, z);
                const c = makeVar('Married',    x, y);
                return impliesAnd(a, b, c);
            })));
}

In [37]:
consistentNames(FirstMale, FirstFemale, SurNames)

{{Married<Erich,Inge>, (¬, FemaleName<Inge,Amann>), (¬, MaleName<Erich,Amann>)}, {Married<Erich,Inge>, (¬, FemaleName<Inge,Bemann>), (¬, MaleName<Erich,Bemann>)}, {Married<Erich,Inge>, (¬, FemaleName<Inge,Cemann>), (¬, MaleName<Erich,Cemann>)}, {Married<Erich,Inge>, (¬, FemaleName<Inge,Demann>), (¬, MaleName<Erich,Demann>)}, {Married<Erich,Josefa>, (¬, FemaleName<Josefa,Amann>), (¬, MaleName<Erich,Amann>)}, {Married<Erich,Josefa>, (¬, FemaleName<Josefa,Bemann>), (¬, MaleName<Erich,Bemann>)}, {Married<Erich,Josefa>, (¬, FemaleName<Josefa,Cemann>), (¬, MaleName<Erich,Cemann>)}, {Married<Erich,Josefa>, (¬, FemaleName<Josefa,Demann>), (¬, MaleName<Erich,Demann>)}, {Married<Erich,Klara>, (¬, FemaleName<Klara,Amann>), (¬, MaleName<Erich,Amann>)}, {Married<Erich,Klara>, (¬, FemaleName<Klara,Bemann>), (¬, MaleName<Erich,Bemann>)}, {Married<Erich,Klara>, (¬, FemaleName<Klara,Cemann>), (¬, MaleName<Erich,Cemann>)}, {Married<Erich,Klara>, (¬, FemaleName<Klara,Demann>), (¬, MaleName<Erich,Demann>)

Now we translate the text constraints natively into TS using our typed logic helpers.

In [56]:
function computeClauses(
    FirstMale:   Set<string>, 
    FirstFemale: Set<string>, 
    SurNames:    Set<string>): Clauses 
{
    let clauses = set<Clause>();
    // Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt.
    clauses = bijective(FirstMale, SurNames, "MaleName");
    // Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt.
    clauses = clauses.union(bijective(FirstMale, FirstFemale, "Married"));
    // Jede Frau hat genau einen Nachnamen
    clauses = clauses.union(bijective(FirstFemale, SurNames, "FemaleName"));
    // Die Namen sind konsistent.
    clauses = clauses.union(consistentNames(FirstMale, FirstFemale, SurNames));
    // Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge.
    clauses = clauses.union(exclusiveOr(makeVar("MaleName", "Heiner", "Amann"), makeVar("FemaleName", "Inge", "Bemann")));
    // Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle – heisst Klaras Mann nicht Fritz.
    clauses = clauses.union(equivalence(makeVar("FemaleName", "Josefa", "Cemann"), complement(makeVar("Married", "Fritz", "Klara"))));
    // Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet.
    clauses = clauses.union(implies(complement(makeVar("Married", "Erich", "Josefa")), makeVar("Married", "Fritz", "Inge")));
    // Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav.
    clauses = clauses.union(implies(makeVar("Married", "Fritz", "Luise"), complement(makeVar("Married", "Gustav", "Klara"))));
    // Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet.
    clauses = clauses.union(implies(makeVar("Married", "Fritz", "Inge"), complement(makeVar("Married", "Erich", "Josefa"))));
    // Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara.
    clauses = clauses.union(implies(complement(makeVar("Married", "Fritz", "Luise")), makeVar("Married", "Gustav", "Klara")));
    // Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav.
    clauses = clauses.union(exclusiveOr(makeVar("FemaleName", "Luise", "Demann"), makeVar("MaleName", "Gustav", "Cemann")));
    return clauses;
}

In [57]:
const Clauses = computeClauses(FirstMale, FirstFemale, SurNames);
Clauses

{{(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Inge,Bemann>)}, {(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Inge,Cemann>)}, {(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Inge,Demann>)}, {(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Josefa,Amann>)}, {(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Klara,Amann>)}, {(¬, FemaleName<Inge,Amann>), (¬, FemaleName<Luise,Amann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, FemaleName<Inge,Cemann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, FemaleName<Inge,Demann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, FemaleName<Josefa,Bemann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, FemaleName<Klara,Bemann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, FemaleName<Luise,Bemann>)}, {(¬, FemaleName<Inge,Bemann>), (¬, MaleName<Heiner,Amann>)}, {(¬, FemaleName<Inge,Cemann>), (¬, FemaleName<Inge,Demann>)}, {(¬, FemaleName<Inge,Cemann>), (¬, FemaleName<Josefa,Cemann>)}, {(¬, FemaleName<Inge,Cemann>), (¬, FemaleName<Klara,Cemann>)}, {(¬, FemaleName<Inge,Cemann>), (¬, FemaleName<Luise,Cemann>)}, {(¬, 

There are 242 different clauses.

In [58]:
Clauses.size

242


In [59]:
function compute_solution(FirstMale: Set<string>, FirstFemale: Set<string>, SurNames: Set<string>): Clauses {
    const systemClauses = computeClauses(FirstMale, FirstFemale, SurNames);
    return solve(systemClauses);
}

In [60]:
const Solution = compute_solution(FirstMale, FirstFemale, SurNames);

In [61]:
function arb<T extends Value>(S: Set<T>): T {
    // Using recursive-set's efficient random element access
    return S.pickRandom()!;
}

In [62]:
function onlyPositive(Solution: Clauses): Set<string> {
    const result = set<string>();
    for (const clause of Solution) {
        const literal = arb(clause);
        if (typeof literal === 'string') {
            result.add(literal);
        }
    }
    return result;
}

In [63]:
onlyPositive(Solution)

{FemaleName<Inge,Bemann>, FemaleName<Josefa,Cemann>, FemaleName<Klara,Amann>, FemaleName<Luise,Demann>, MaleName<Erich,Demann>, MaleName<Fritz,Bemann>, MaleName<Gustav,Amann>, MaleName<Heiner,Cemann>, Married<Erich,Luise>, Married<Fritz,Inge>, Married<Gustav,Klara>, Married<Heiner,Josefa>}


In [64]:
function extractFirst(s: string): string {
    const m = s.match(/<([A-Za-z]+),/);
    return m ? m[1] : '';
}

function extractSecond(s: string): string {
    const m = s.match(/,([A-Za-z]+)>/);
    return m ? m[1] : '';
}

In [65]:
function displaySolution(Solution: Clauses) {
    const married: Record<string, string> = {};
    const names: Record<string, string> = {};
    
    for (const unit of Solution) {
        for (const l of unit) {
            if (typeof l === 'string') {
                if (l.startsWith("Married")) {
                    married[extractFirst(l)] = extractSecond(l);
                } else if (l.startsWith("MaleName")) {
                    names[extractFirst(l)] = extractSecond(l);
                }
            }
        }
    }
    for (const x of Object.keys(married)) {
        console.log(`${x} ${names[x]} is married to ${married[x]}.`);
    }
}

In [66]:
displaySolution(Solution);

Gustav Amann is married to Klara.
Fritz Bemann is married to Inge.
Erich Demann is married to Luise.
Heiner Cemann is married to Josefa.


## Checking the Uniqueness of the Solution

Given a set of unit clauses $U$, the function `checkUniqueness(U)` returns a clause that is the negation of the set $U$.

In [67]:
function negateSolution(UnitClauses: Clauses): Set<Literal> {
    const result = set<Literal>();
    for (const unit of UnitClauses) {
        result.add(complement(arb(unit)));
    }
    return result;
}

In [68]:
negateSolution(set<Clause>(set<Literal>('a'), set<Literal>(tpl('¬', 'b'))))

{b, (¬, a)}


In [69]:
function checkUniqueness(Solution: Clauses, baseClauses: Clauses) {
    const negation         = negateSolution(Solution);
    const augmentedClauses = baseClauses.union(set<Clause>(negation));
    
    const alternative = solve(augmentedClauses);
    const emptyClause = set<Literal>();
    
    // If alternative contains the empty clause {{}}, it's inconsistent
    if (alternative.has(emptyClause) && alternative.size === 1) {
        console.log("Well done: The solution is unique!");
    } else {
        console.log("ERROR: The solution is not unique!");
    }
}

In [70]:
checkUniqueness(Solution, Clauses);

Well done: The solution is unique!
